---
title: "Receiver calibration"
bibliography: ../bibliography.bib
---


# Table of Contents

1. [Receiver calibration](#receiver-calibration)
2. [Why receiver DCB matters](#why-receiver-dcb-matters)
3. [Calibration as model alignment](#calibration-as-model-alignment)
4. [PPP-UDUC as an advanced DCB source](#ppp-uduc-as-an-advanced-dcb-source)
5. [Current GNX-py calibration workflow](#current-gnx-py-calibration-workflow)
6. [Notebook setup](#notebook-setup)
7. [Weighted-average receiver calibration](#weighted-average-receiver-calibration)
8. [Comparing calibrated, GIM and user-defined DCB](#comparing-calibrated-gim-and-user-defined-dcb)
9. [How to pass your own receiver DCB](#how-to-pass-your-own-receiver-dcb)
10. [What to try next](#what-to-try-next)


# Receiver calibration

In this tutorial, I will discuss the issue of calibrating GNSS receivers for TEC measurement. Several scientific articles have been devoted to this topic, and they are worth reading for a deeper understanding of the subject [@Sardon1994; @Arikan2008]. Here we will keep the same style as in the previous ionosphere notebook: first the observation model, then the intuition, then a small GNX-py example.

The practical question is simple: when we form a geometry-free code combination, the ionospheric delay is exposed, but so are satellite and receiver hardware delays. If the receiver delay is unknown, the STEC time series can be shifted by a constant amount. For relative short-term variability that may be acceptable; for absolute STEC it is not.


# Why receiver DCB matters

Let us start from code observations at two frequencies:

$$
P_1 = \rho + c(\delta t_r - \delta t_s) + T + I_1 + b^{P}_{r,1} + b^{P}_{s,1} + \varepsilon_{P_1} \quad [m]
$$

$$
P_2 = \rho + c(\delta t_r - \delta t_s) + T + I_2 + b^{P}_{r,2} + b^{P}_{s,2} + \varepsilon_{P_2} \quad [m]
$$

and the geometry-free code combination:

$$
P_{\mathrm{GF}} = P_1 - P_2 = (I_1 - I_2) + (b^{P}_{r,1}-b^{P}_{r,2}) + (b^{P}_{s,1}-b^{P}_{s,2}) + (\varepsilon_{P_1}-\varepsilon_{P_2}) \quad [m].
$$

where:

- $P_1$, $P_2$ are code observations on two frequencies,
- $P_{\mathrm{GF}}$ is the geometry-free code combination,
- $I_n$ is the ionospheric delay in metres at frequency $n$,
- $b^{P}_{r,n}$ and $b^{P}_{s,n}$ are receiver and satellite hardware delays.

The non-dispersive range, clocks and troposphere disappeared, but the hardware delays remained. In other words, the geometry-free combination is exactly the place where the ionosphere becomes visible and where DCB errors become dangerous.

During the generation of global ionosphere maps, receiver and satellite biases are estimated as part of the state vector. That is why IONEX/GIM files often contain satellite and receiver DCB entries in the header. You can also find SINEX BIAS products with `.BIA` or `.BSX` extensions, containing DSB or OSB values for satellites and sometimes receivers [@SINEXBIAS]. What we need for absolute STEC is the bias in the same geometry-free convention used by our selected signal pair.

**Current version note.** GNX-py now exposes this explicitly through the STEC bias policy. `TECConfig.stec_bias_sources` chooses the fallback order, while output columns such as `stec_sat_bias_source`, `stec_station_bias_source`, `stec_bias_code_1` and `stec_bias_code_2` tell us which convention was actually used.


# Calibration as model alignment

The receiver calibration method used by GNX-py can be summarized as levelling the measured geometry-free ionosphere to a background model. This is similar in spirit to phase-to-code levelling from the previous notebook, but now the unknown constant is the receiver DCB.

In the weighted-average method, GNX-py uses a background GIM value and satellite DCB information. For each satellite link above an elevation cutoff, it forms an epoch-wise receiver DCB candidate. In simplified notation:

$$
\hat{d}_{r}(t, s) = I^{\mathrm{model}}_{s}(t) - \left(L_4(t,s)/c + d^{sat}_{12}(s)\right),
$$

where $L_4$ is the levelled geometry-free phase observable expressed as a time delay, $d^{sat}_{12}$ is the satellite differential code bias in seconds, and $I^{\mathrm{model}}$ is the modelled ionospheric delay converted to the same convention.

The epoch value is an elevation-weighted average:

$$
\hat{d}_{r}(t) = \frac{\sum_s w_s(t)\hat{d}_{r}(t,s)}{\sum_s w_s(t)}, \qquad
w_s(t)=\sin^4(E_s(t)).
$$

Finally, the receiver DCB is taken as a robust median over epochs:

$$
\hat{d}_{r} = \mathrm{median}_t\left(\hat{d}_{r}(t)\right).
$$

The exact sign convention follows GNX-py's internal `P4/L4` convention and the selected signal pair. The important practical point is consistency: GIM model, satellite bias, receiver bias and geometry-free observable must all be interpreted in the same convention.

This method is model-dependent. If the background GIM is biased, the receiver DCB estimate will inherit part of that bias. That is why GIM is a reasonable background for this tutorial, while Klobuchar or NTCM would be too rough for calibration.


# PPP-UDUC as an advanced DCB source

The older version of this notebook also compared the GIM-based weighted calibration with a PPP-UDUC estimate stored in a local PPP output CSV. That file is not part of the current `sample_data`, so the PPP comparison is kept here as a workflow note rather than a mandatory executable cell.

The idea remains useful. In uncombined PPP, receiver hardware delays can be part of the estimated state. If one observable-specific bias is fixed as a reference, the second one can be interpreted as a DCB-like quantity for the receiver. A helper such as a `ppp2dsb` post-processing script can turn PPP output into a station DCB table. You can then pass such a value back into `TECConfig` through `rcv_dcb_source="defined"` and `define_station_dcb=...`.

This PPP route can be more robust than a simple GIM alignment for some stations, but it is also more complex and still depends on the chosen ionospheric constraints, bias products and PPP convergence. In this tutorial we keep the executable part focused on the current GIM-based calibration path and show how a user-defined DCB would be injected.


# Current GNX-py calibration workflow

In the current API, the automatic receiver calibration path is activated through `TECConfig`:

```python
add_sta_dcb=True
rcv_dcb_source="calibrate"
```

Then `TECSession` does two passes internally:

1. It temporarily disables station DCB correction and measures preliminary STEC.
2. It runs `Calibration` against the GIM background to estimate a receiver DCB.
3. It stores the estimate in `sta_dcb` and reprocesses STEC using the normal bias policy.

Other receiver-bias choices are also available:

| `rcv_dcb_source` | meaning |
|---|---|
| `"calibrate"` | estimate station DCB from measured STEC and GIM background |
| `"gim"` | use receiver DCB parsed from GIM/IONEX auxiliary data |
| `"defined"` | use `define_station_dcb` from the configuration |
| `"none"` | do not apply station DCB |

This makes the calibration notebook part of the same bias-policy story as the introduction notebook. We do not hide the chosen DCB source; we inspect it.


# Notebook setup

The previous version used BOR1 paths and a PPP output file outside the repository. The updated notebook uses `BRUX` from `sample_data`, because that is the station currently included with the repository. The calculations avoid local absolute paths and do not write output files.


In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import contextlib
import importlib.util
import io
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import gnx_py as gnx
from gnx_py.gnss import frequency_hz, mode_signals
from gnx_py.ionosphere import TECConfig, TECSession


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "gnx_py").is_dir():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the cloned GNX-py repository, "
        "or set REPO_ROOT manually to the repository directory."
    )


REPO_ROOT = find_repo_root(Path.cwd())
SAMPLE_DATA = REPO_ROOT / "sample_data"
TUTORIAL_DIR = REPO_ROOT / "tutorial" / "ionosphere"

print(f"gnx_py imported from: {gnx.__file__}")
print(f"sample data: {SAMPLE_DATA}")


In [ ]:
def load_tutorial_module(module_name: str, module_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError(f"Cannot load {module_name} from {module_path}")
    spec.loader.exec_module(module)
    return module


tools = load_tutorial_module("ionosphere_tutorial_tools", TUTORIAL_DIR / "tools.py")
df_head = tools.df_head


In [ ]:
STATION = "BRUX"
SYSTEM = "G"
GPS_MODE = "L1L2"
DOY = 35
C = 299_792_458.0

OBS_PATH = SAMPLE_DATA / "BRUX00BEL_R_20240350000_01D_30S_MO.crx.gz"
NAV_PATH = SAMPLE_DATA / "BRDC00IGS_R_20240350000_01D_MN.rnx"
SP3_PATHS = [
    SAMPLE_DATA / "GRG0MGXFIN_20240340000_01D_05M_ORB.SP3",
    SAMPLE_DATA / "GRG0MGXFIN_20240350000_01D_05M_ORB.SP3",
    SAMPLE_DATA / "GRG0MGXFIN_20240360000_01D_05M_ORB.SP3",
]
ATX_PATH = SAMPLE_DATA / "igs20.atx"
DCB_PATH = SAMPLE_DATA / "CAS1OPSRAP_20240350000_01D_01D_DCB.BIA"
GIM_PATH = SAMPLE_DATA / "COD0OPSFIN_20240350000_01D_01H_GIM.INX"
PPP_DCB_TABLE = SAMPLE_DATA / "dcb_table.csv"

# TECSession currently calibrates against the available GIM arc data; this short
# limit keeps the user-facing intent clear, even though calibration may use the
# product coverage needed internally by the current implementation.
TLIM = [datetime(2024, 2, 4, 0, 0), datetime(2024, 2, 4, 0, 30)]

required_paths = [OBS_PATH, NAV_PATH, ATX_PATH, DCB_PATH, GIM_PATH, PPP_DCB_TABLE, *SP3_PATHS]
missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing sample files:\n" + "\n".join(str(path) for path in missing))


In [ ]:
def run_tec(config: TECConfig, use_sys: str = SYSTEM):
    """Run TECSession while keeping notebook output compact."""
    stream = io.StringIO()
    with warnings.catch_warnings(record=True) as captured, contextlib.redirect_stdout(stream):
        warnings.simplefilter("always")
        result = TECSession(config=config, use_sys=use_sys).run()
    warning_messages = list(dict.fromkeys(str(item.message) for item in captured))
    return result, warning_messages


def build_calibration_config(
    *,
    rcv_dcb_source: str = "calibrate",
    define_station_dcb=None,
    add_sta_dcb: bool = True,
) -> TECConfig:
    return TECConfig(
        obs_path=OBS_PATH,
        nav_path=NAV_PATH,
        sp3_path=SP3_PATHS,
        atx_path=ATX_PATH,
        dcb_path=DCB_PATH,
        gim_path=GIM_PATH,
        sys=SYSTEM,
        gps_freq=GPS_MODE,
        gal_freq="E1E5a",
        bds_freq="B1IB3I",
        orbit_type="broadcast",
        ionosphere_model="gim",
        troposphere_model=False,
        time_limit=TLIM,
        day_of_year=DOY,
        station_name=STATION,
        screen=False,
        ev_mask=30,
        use_gfz=True,
        windup=True,
        rel_path=False,
        sat_pco=False,
        rec_pco=True,
        add_dcb=True,
        add_sta_dcb=add_sta_dcb,
        rcv_dcb_source=rcv_dcb_source,
        define_station_dcb=define_station_dcb,
        stec_bias_sources=("product", "gim", "config", "zero"),
        stec_missing_bias="warn_zero",
        compare_models=False,
        use_iono_rms=True,
        leveling_ws=20,
        median_leveling_ws=20,
        min_arc_len=2,
    )


def tec_factor_for_mode(mode: str) -> float:
    sig1, sig2 = mode_signals(mode)
    f1, f2 = frequency_hz(sig1), frequency_hz(sig2)
    return (1 / 40.3) * ((f1**2 * f2**2) / (f1**2 - f2**2))


TEC_FACTOR = tec_factor_for_mode(GPS_MODE)
NS_TO_TECU = C * 1e-9 * TEC_FACTOR / 1e16


def to_tecu(series: pd.Series) -> pd.Series:
    return series / 1e16


# Weighted-average receiver calibration

We now run the current GNX-py calibration path. The important configuration choices are `add_sta_dcb=True` and `rcv_dcb_source="calibrate"`. The result dataframe should contain both the GIM receiver bias (`sta_bias`) and the calibrated receiver bias (`sta_dcb`).


In [ ]:
calibration_config = build_calibration_config(rcv_dcb_source="calibrate")
obs_tec, calibration_warnings = run_tec(calibration_config)

print(f"rows={len(obs_tec)}")
print(f"satellite arcs={obs_tec.index.get_level_values('sv').nunique()}")
print(f"unique captured warnings={len(calibration_warnings)}")


In [ ]:
calibrated_bias_ns = float(obs_tec["sta_dcb"].dropna().iloc[0])
gim_bias_ns = float(obs_tec["sta_bias"].dropna().iloc[0])
bias_error_ns = gim_bias_ns - calibrated_bias_ns

bias_table = pd.DataFrame(
    [
        {"source": "GIM / IONEX header", "receiver_dcb_ns": gim_bias_ns},
        {"source": "GNX weighted calibration", "receiver_dcb_ns": calibrated_bias_ns},
    ]
)
bias_table["difference_to_gim_ns"] = bias_table["receiver_dcb_ns"] - gim_bias_ns
bias_table["stec_shift_vs_calibrated_tecu"] = (
    bias_table["receiver_dcb_ns"] - calibrated_bias_ns
) * NS_TO_TECU

df_head(bias_table, nrows=5, ncols=4, floatfmt=".3f", reset_index=False)
print(f"Calibration difference for station {STATION}: {bias_error_ns:.3f} ns (GIM - calibrated)")


A difference below about one nanosecond is generally a good sign for this simple method. In STEC units, the same difference is not abstract: for GPS L1/L2 it maps to a nearly constant shift in TECU. This is why receiver calibration matters even when the shape of the STEC series looks correct.


In [ ]:
diagnostic_cols = [
    "sta_bias", "sta_dcb", "stec_station_bias_source", "stec_sat_bias_source",
    "stec_bias_code_1", "stec_bias_code_2", "stec_station_bias_m",
    "stec_sat_bias_m", "stec_total_bias_m", "leveled_tec", "ev",
]
diagnostic_cols = [column for column in diagnostic_cols if column in obs_tec.columns]
df_head(obs_tec[diagnostic_cols], nrows=5, ncols=11, floatfmt=".3f", truncate_str=16)


The diagnostic columns confirm which path was used. For the calibrated run, `stec_station_bias_source` should be `calibrate`. Satellite biases usually come from the product/GIM path depending on the selected signal pair and available columns.


# Comparing calibrated, GIM and user-defined DCB

The old notebook reprocessed STEC three times: calibrated DCB, GIM DCB and PPP-UDUC DCB. To keep this version light and self-contained, we use the calibrated STEC as the base series and compute the constant STEC shift implied by alternative receiver DCB values. This is equivalent for illustrating the main effect: changing a receiver DCB shifts the absolute STEC level.

The formula used in the next cell is:

$$
\Delta \mathrm{STEC}[\mathrm{TECU}] = \Delta d_r[\mathrm{ns}] \cdot c \cdot 10^{-9} \cdot \frac{K_{GF}}{10^{16}},
$$

where $K_{GF}$ is the geometry-free conversion factor for the selected frequency pair.


In [ ]:
arc_counts = obs_tec.groupby(level="sv").size().sort_values()
selected_arc = arc_counts.idxmax()
base_stec = to_tecu(obs_tec.loc[selected_arc, "leveled_tec"]).reset_index()

series = pd.DataFrame({"time": base_stec["time"]})
series["calibrated"] = base_stec["leveled_tec"]
series["gim_header"] = series["calibrated"] + (gim_bias_ns - calibrated_bias_ns) * NS_TO_TECU
series["no_station_dcb"] = series["calibrated"] + (0.0 - calibrated_bias_ns) * NS_TO_TECU

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(series["time"], series["calibrated"], label="Weighted calibration", s=5, c="tab:red", marker="o")
ax.scatter(series["time"], series["gim_header"], label="GIM receiver DCB", s=5, c="black", marker="x")
ax.scatter(series["time"], series["no_station_dcb"], label="No station DCB", s=5, c="tab:blue", marker=".")
ax.set_title(f"Receiver DCB effect on STEC for arc {selected_arc}")
ax.set_ylabel("STEC [TECU]")
ax.set_xlabel("Time")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend()
plt.tight_layout()

print(f"Selected arc: {selected_arc}")


In [ ]:
shift_table = pd.DataFrame(
    [
        {"series": "GIM header vs calibrated", "shift_tecu": (gim_bias_ns - calibrated_bias_ns) * NS_TO_TECU},
        {"series": "No station DCB vs calibrated", "shift_tecu": (0.0 - calibrated_bias_ns) * NS_TO_TECU},
    ]
)
df_head(shift_table, nrows=5, ncols=2, floatfmt=".3f", reset_index=False)


This plot is intentionally simple. It shows that receiver DCB acts as an almost constant vertical offset in absolute STEC. That offset may be small when calibrated and GIM values agree, or very large if station DCB is omitted. The shape of the arc can remain similar, while the absolute TEC level becomes wrong.


# PPP-UDUC comparison when available

The repository includes `sample_data/dcb_table.csv`, which is a compact table of PPP-derived DCB values for some stations. The current BRUX sample station is not necessarily present in that table. The cell below is therefore diagnostic: it uses a PPP-derived value if one exists for `STATION`, otherwise it explains what the user should provide.


In [ ]:
ppp_table = pd.read_csv(PPP_DCB_TABLE)
ppp_match = ppp_table.loc[ppp_table["Station"].str.upper().eq(STATION)]

if ppp_match.empty:
    print(
        f"No PPP-derived DCB row for {STATION} in {PPP_DCB_TABLE.name}. "
        "To compare PPP-UDUC here, provide a station DCB value and pass it as define_station_dcb."
    )
else:
    ppp_bias_ns = float(ppp_match["gps_dcb_mean"].iloc[0])
    ppp_shift_tecu = (ppp_bias_ns - calibrated_bias_ns) * NS_TO_TECU
    ppp_summary = pd.DataFrame(
        [{
            "source": "PPP-UDUC table mean",
            "receiver_dcb_ns": ppp_bias_ns,
            "difference_to_calibrated_ns": ppp_bias_ns - calibrated_bias_ns,
            "stec_shift_vs_calibrated_tecu": ppp_shift_tecu,
        }]
    )
    df_head(ppp_summary, nrows=1, ncols=4, floatfmt=".3f", reset_index=False)


# How to pass your own receiver DCB

When you have a trusted receiver DCB, for example from PPP-UDUC or from an external calibration product, you can pass it through `TECConfig` as a user-defined station bias. The value is in nanoseconds.

The cell below builds the configuration but does not run another full STEC session by default. It is meant as a clear template that you can enable after replacing `user_receiver_dcb_ns` with your own value.


In [ ]:
user_receiver_dcb_ns = calibrated_bias_ns

defined_config = build_calibration_config(
    rcv_dcb_source="defined",
    define_station_dcb=user_receiver_dcb_ns,
    add_sta_dcb=True,
)

print(f"rcv_dcb_source={defined_config.rcv_dcb_source}")
print(f"define_station_dcb={defined_config.define_station_dcb:.3f} ns")

RUN_DEFINED_EXAMPLE = False
if RUN_DEFINED_EXAMPLE:
    obs_tec_defined, defined_warnings = run_tec(defined_config)
    print(f"defined run rows={len(obs_tec_defined)} warnings={len(defined_warnings)}")


For validation runs, consider setting `stec_missing_bias="raise"`. That makes missing bias products fail loudly instead of silently falling back to zero. During exploration, `warn_zero` is convenient; during scientific reporting, explicit failure is usually safer.


# What to try next

This concludes the updated calibration tutorial. We preserved the main idea from the original notebook: receiver DCB calibration is a levelling problem, and the selected ionosphere model matters. The executable part now uses current `sample_data`, current `TECConfig`, and explicit bias-policy diagnostics.

The main lessons are:

- a receiver DCB error produces an almost constant STEC offset,
- `rcv_dcb_source="calibrate"` estimates the station bias against the GIM background,
- `rcv_dcb_source="gim"` uses receiver DCB from the GIM/IONEX header,
- `rcv_dcb_source="defined"` is the path for PPP-derived or externally supplied DCB values,
- PPP-UDUC comparison is still relevant, but it needs a current PPP output or station DCB table for the same station.

The next tutorial priority after this one is not `activity_indexes` or `kriging`; those remain deferred and data-dependent. A good next step is a pass over the tutorial set as a whole, or a focused update of whichever non-deferred notebook you want to polish next.
